In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    regexp_replace,
    when
)

spark = SparkSession.builder \
    .appName("DataCleaningWranglingDemo") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)

In [ ]:
data = [
    (101, "  John Smith  ", "IT",       " JOHN.SMITH@ABC.COM ", "987-654-3210", 60000),
    (102, "Mary Johnson",  " hr ",      "mary.johnson@abc.com", "9876543211",    55000),
    (103, "  David Brown",  "FINANCE",   "DAVID.BROWN@ABC.COM",  "987 654 3212",  70000),
    (103, "  David Brown",  "FINANCE",   "DAVID.BROWN@ABC.COM",  "987 654 3212",  70000),
    (104, "Lisa Wilson ",   None,        "lisa.wilson@abc.com",   "987-654-3213",  None),
    (109, "Lisa Wilson ",   None,        "lisa.wilson@abc.com",   "987-654-3213",  None),
    (105, None,             "IT",        "james@abc",             "987-654-3214",  62000),
    (106, "Robert Taylor",  "it",        "robert.taylor@abc.com", "9876543215",    65000),
    (107, "  Susan Lee ",   " HR",       None,                    "987-654-3216",  58000),
    (108, "Peter Adams",    "Finance ",  "peter.adams@abc.com",   None,            72000)
]

columns = [
    "emp_id",
    "name",
    "department",
    "email",
    "phone",
    "salary"
]

df = spark.createDataFrame(data, columns)

df.show(truncate=False)

In [ ]:
df.printSchema()

In [ ]:
df.show(truncate=True)

In [ ]:
from pyspark.sql.functions import sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

In [ ]:
df.filter(
    col("name").isNull() |
    col("department").isNull() |
    col("email").isNull() |
    col("phone").isNull() |
    col("salary").isNull()
).show(truncate=False)

In [ ]:
df_filled = df.fillna({
    "department": "UNKNOWN",
    "salary": 0,
    "email": "NOT_AVAILABLE",
    "phone": "NOT_AVAILABLE"
})

df_filled.show(truncate=False)

In [ ]:
df_valid = df.dropna(
    subset=["emp_id", "name"]
)

df_valid.show(truncate=False)

In [ ]:
df_no_nulls = df.dropna()

df_no_nulls.show(truncate=False)

In [ ]:
df.groupBy(
    "emp_id"
).count().filter(
    col("count") > 1
).show()

In [ ]:
df_distinct = df.distinct()

df_distinct.show(truncate=False)

In [ ]:
df_deduplicated = df.dropDuplicates(["emp_id"])

df_deduplicated.show(truncate=False)

In [ ]:
df_email_check = df.withColumn(
    "email_status",
    when(
        col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"),
        "VALID"
    ).otherwise("INVALID")
)

df_email_check.select(
    "emp_id",
    "name",
    "email",
    "email_status"
).show(truncate=False)

In [ ]:
df_trimmed = df.withColumn(
    "name",
    trim(col("name"))
).withColumn(
    "department",
    trim(col("department"))
)

df_trimmed.show(truncate=False)

In [ ]:
df_lower = df.withColumn(
    "email",
    lower(trim(col("email")))
)

df_lower.select(
    "emp_id",
    "email"
).show(truncate=False)

In [ ]:
df_upper = df.withColumn(
    "department",
    upper(trim(col("department")))
)

df_upper.select(
    "emp_id",
    "department"
).show()

In [ ]:
df_phone_clean = df.withColumn(
    "clean_phone",
    regexp_replace(col("phone"), r"[^0-9]", "")
)

df_phone_clean.select(
    "emp_id",
    "phone",
    "clean_phone"
).show(truncate=False)

In [ ]:
df_name_clean = df.withColumn(
    "clean_name",
    regexp_replace(
        trim(col("name")),
        r"\s+",
        " "
    )
)

df_name_clean.select(
    "emp_id",
    "name",
    "clean_name"
).show(truncate=False)

In [ ]:
clean_df = df \
    .dropDuplicates(["emp_id"]) \
    .fillna({
        "department": "UNKNOWN",
        "email": "NOT_AVAILABLE",
        "phone": "NOT_AVAILABLE",
        "salary": 0
    }) \
    .withColumn(
        "name",
        regexp_replace(
            trim(col("name")),
            r"\s+",
            " "
        )
    ) \
    .withColumn(
        "department",
        upper(trim(col("department")))
    ) \
    .withColumn(
        "email",
        lower(trim(col("email")))
    ) \
    .withColumn(
        "phone",
        regexp_replace(col("phone"), r"[^0-9]", "")
    )

clean_df.show(truncate=False)

In [ ]:
clean_df.groupBy("emp_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [ ]:
clean_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in clean_df.columns
]).show()

In [ ]:
clean_df.select(
    "emp_id",
    "name",
    "department",
    "email",
    "phone",
    "salary"
).show(truncate=False)

In [ ]:
clean_df.withColumn(
    "email_status",
    when(
        col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        "VALID"
    ).otherwise("INVALID")
).select(
    "emp_id",
    "email",
    "email_status"
).show(truncate=False)

In [ ]:
print("Total Records:", df.count())

print(
    "Unique Employee IDs:",
    df.select("emp_id").distinct().count()
)

print(
    "Duplicate Employee IDs:",
    df.groupBy("emp_id")
      .count()
      .filter(col("count") > 1)
      .count()
)

In [ ]:
print("Cleaned Records:", clean_df.count())

print(
    "Cleaned Unique Employee IDs:",
    clean_df.select("emp_id").distinct().count()
)